# Tensors

Tensors are a specialized data structure that are very similar to arrays
and matrices. In PyTorch, we use tensors to encode the inputs and
outputs of a model, as well as the model's parameters.

Tensors are similar to [NumPy's](https://numpy.org/) ndarrays, except
that tensors can run on GPUs or other hardware accelerators. In fact,
tensors and NumPy arrays can often share the same underlying memory,
eliminating the need to copy data. Tensors are also
optimized for automatic differentiation. If you're
familiar with ndarrays, you'll be right at home with the Tensor API. If
not, follow along!

In [ ]:
import torch
import numpy as np

## Initializing a Tensor

Tensors can be initialized in various ways. Take a look at the following
examples:

**Directly from data**

Tensors can be created directly from data. The data type is
automatically inferred.


In [ ]:
data = [[1, 2],[3, 1]]
x_data = torch.tensor(data)

**From a NumPy array**

Tensors can be created from NumPy arrays.


In [ ]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)

**From another tensor:**

The new tensor retains the properties (shape, datatype) of the argument
tensor, unless explicitly overridden.


In [ ]:
x_data.shape

torch.Size([2, 2])

In [ ]:
x_ones = torch.ones_like(x_data) # retains the properties of x_data
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # overrides the datatype of x_data
print(f"Random Tensor: \n {x_rand} \n")

Ones Tensor: 
 tensor([[1, 1],
        [1, 1]]) 

Random Tensor: 
 tensor([[0.3553, 0.8686],
        [0.1750, 0.8328]]) 



**With random or constant values:**

`shape` is a tuple of tensor dimensions. In the functions below, it
determines the dimensionality of the output tensor.


In [ ]:
shape = (2,3,)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

Random Tensor: 
 tensor([[0.7311, 0.0626, 0.8145],
        [0.5394, 0.8733, 0.3181]]) 

Ones Tensor: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Zeros Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])


------------------------------------------------------------------------


## Attributes of a Tensor

Tensor attributes describe their shape, datatype, and the device on
which they are stored.


In [ ]:
tensor = torch.rand(3,4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


------------------------------------------------------------------------


## Operations on Tensors

Over 1200 tensor operations, including arithmetic, linear algebra,
matrix manipulation (transposing, indexing, slicing), sampling and more
are comprehensively described
[here](https://pytorch.org/docs/stable/torch.html).

Each of these operations can be run on the CPU and
[Accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If you're using Colab, allocate an
accelerator by going to Runtime \> Change runtime type \> GPU.

By default, tensors are created on the CPU. We need to explicitly move
tensors to the accelerator using `.to` method (after checking for
accelerator availability). Keep in mind that copying large tensors
across devices can be expensive in terms of time and memory!


In [ ]:
# We move our tensor to the current accelerator if available
if torch.accelerator.is_available():
    tensor = tensor.to(torch.accelerator.current_accelerator())

In [ ]:
tensor.device

device(type='cuda', index=0)

Try out some of the operations from the list. If you\'re familiar with
the NumPy API, you\'ll find the Tensor API a breeze to use.


**Standard numpy-like indexing and slicing:**


In [ ]:
tensor = torch.ones(4, 4)
print(f"First row: {tensor[0]}")
print(f"First column: {tensor[:, 0]}")
print(f"Last column: {tensor[..., -1]}")
tensor[:,1] = 0
print(tensor)

First row: tensor([1., 1., 1., 1.])
First column: tensor([1., 1., 1., 1.])
Last column: tensor([1., 1., 1., 1.])
tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


**Joining tensors** You can use `torch.cat` to concatenate a sequence of
tensors along a given dimension. See also
[torch.stack](https://pytorch.org/docs/stable/generated/torch.stack.html),
another tensor joining operator that is subtly different from
`torch.cat`.


In [ ]:
t1 = torch.cat([tensor, tensor, tensor], dim=0)
print(t1)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


**Arithmetic operations**


In [ ]:
# This computes the matrix multiplication between two tensors. y1, y2, y3 will have the same value
# ``tensor.T`` returns the transpose of a tensor
y1 = tensor @ tensor.T
y2 = tensor.matmul(tensor.T)

y3 = torch.rand_like(y1)
torch.matmul(tensor, tensor.T, out=y3)


# This computes the element-wise product. z1, z2, z3 will have the same value
z1 = tensor * tensor
z2 = tensor.mul(tensor)

z3 = torch.rand_like(tensor)
torch.mul(tensor, tensor, out=z3)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])

**Single-element tensors** If you have a one-element tensor, for example
by aggregating all values of a tensor into one value, you can convert it
to a Python numerical value using `item()`:


In [ ]:
agg = tensor.sum()
agg_item = agg.item()
print(agg_item, type(agg_item))

12.0 <class 'float'>


**In-place operations** Operations that store the result into the
operand are called in-place. They are denoted by a `_` suffix. For
example: `x.copy_(y)`, `x.t_()`, will change `x`.


In [ ]:
print(f"{tensor} \n")
tensor.add_(5)
print(tensor)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]]) 

tensor([[6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.]])


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>In-place operations save some memory, but can be problematic when computing derivatives because of an immediate lossof history. Hence, their use is discouraged.</p>

</div>



------------------------------------------------------------------------


## Bridge with NumPy

Tensors on the CPU and NumPy arrays can share their underlying memory
locations, and changing one will change the other.


## Tensor to NumPy array


In [ ]:
t = torch.ones(5)
print(f"t: {t}")
n = t.numpy()
print(f"n: {n}")

t: tensor([1., 1., 1., 1., 1.])
n: [1. 1. 1. 1. 1.]


A change in the tensor reflects in the NumPy array.


In [ ]:
t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.])
n: [2. 2. 2. 2. 2.]


## NumPy array to Tensor


In [ ]:
n = np.ones(5)
t = torch.from_numpy(n)

Changes in the NumPy array reflects in the tensor.


In [ ]:
np.add(n, 1, out=n)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.], dtype=torch.float64)
n: [2. 2. 2. 2. 2.]


# Automatic Differentiation with `torch.autograd`

When training neural networks, the most frequently used algorithm is
**back propagation**. In this algorithm, parameters (model weights) are
adjusted according to the **gradient** of the loss function with respect
to the given parameter.

To compute those gradients, PyTorch has a built-in differentiation
engine called `torch.autograd`. It supports automatic computation of
gradient for any computational graph.

Consider the simplest one-layer neural network, with input `x`,
parameters `w` and `b`, and some loss function. It can be defined in
PyTorch in the following manner:


In [ ]:
import torch

x = torch.ones(5)  # input tensor
y = torch.zeros(3)  # expected output
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

## Tensors, Functions and Computational graph

This code defines the following **computational graph**:

![](https://pytorch.org/tutorials/_static/img/basics/comp-graph.png)

In this network, `w` and `b` are **parameters**, which we need to
optimize. Thus, we need to be able to compute the gradients of loss
function with respect to those variables. In order to do that, we set
the `requires_grad` property of those tensors.


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>You can set the value of <code>requires_grad</code> when creating atensor, or later by using <code>x.requires_grad_(True)</code> method.</p>

</div>



A function that we apply to tensors to construct computational graph is
in fact an object of class `Function`. This object knows how to compute
the function in the *forward* direction, and also how to compute its
derivative during the *backward propagation* step. A reference to the
backward propagation function is stored in `grad_fn` property of a
tensor. You can find more information of `Function` [in the
documentation](https://pytorch.org/docs/stable/autograd.html#function).


In [ ]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x7fc9d05b3f40>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x7fc9d05b3f40>


## Computing Gradients

To optimize weights of parameters in the neural network, we need to
compute the derivatives of our loss function with respect to parameters,
namely, we need $\frac{\partial loss}{\partial w}$ and
$\frac{\partial loss}{\partial b}$ under some fixed values of `x` and
`y`. To compute those derivatives, we call `loss.backward()`, and then
retrieve the values from `w.grad` and `b.grad`:


In [ ]:
loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.0277, 0.0588, 0.1132],
        [0.0277, 0.0588, 0.1132],
        [0.0277, 0.0588, 0.1132],
        [0.0277, 0.0588, 0.1132],
        [0.0277, 0.0588, 0.1132]])
tensor([0.0277, 0.0588, 0.1132])


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<ul>
<li>We can only obtain the <code>grad</code> properties for the leafnodes of the computational graph, which have <code>requires_grad</code> propertyset to <code>True</code>. For all other nodes in our graph, gradients will not beavailable.- We can only perform gradient calculations using<code>backward</code> once on a given graph, for performance reasons. If we needto do several <code>backward</code> calls on the same graph, we need to pass<code>retain_graph=True</code> to the <code>backward</code> call.</li>
</ul>
```

</div>



## Disabling Gradient Tracking

By default, all tensors with `requires_grad=True` are tracking their
computational history and support gradient computation. However, there
are some cases when we do not need to do that, for example, when we have
trained the model and just want to apply it to some input data, i.e. we
only want to do *forward* computations through the network. We can stop
tracking computations by surrounding our computation code with
`torch.no_grad()` block:


In [ ]:
z = torch.matmul(x, w)+b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

True
False


Another way to achieve the same result is to use the `detach()` method
on the tensor:


In [ ]:
z = torch.matmul(x, w)+b
z_det = z.detach()
print(z_det.requires_grad)

False


There are reasons you might want to disable gradient tracking:

:   -   To mark some parameters in your neural network as **frozen
        parameters**.
    -   To **speed up computations** when you are only doing forward
        pass, because computations on tensors that do not track
        gradients would be more efficient.


## More on Computational Graphs

Conceptually, autograd keeps a record of data (tensors) and all executed
operations (along with the resulting new tensors) in a directed acyclic
graph (DAG) consisting of
[Function](https://pytorch.org/docs/stable/autograd.html#torch.autograd.Function)
objects. In this DAG, leaves are the input tensors, roots are the output
tensors. By tracing this graph from roots to leaves, you can
automatically compute the gradients using the chain rule.

In a forward pass, autograd does two things simultaneously:

-   run the requested operation to compute a resulting tensor
-   maintain the operation's *gradient function* in the DAG.

The backward pass kicks off when `.backward()` is called on the DAG
root. `autograd` then:

-   computes the gradients from each `.grad_fn`,
-   accumulates them in the respective tensor's `.grad` attribute
-   using the chain rule, propagates all the way to the leaf tensors.

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>An important thing to note is that the graph is recreated from scratch; after each<code>.backward()</code> call, autograd starts populating a new graph. This isexactly what allows you to use control flow statements in your model;you can change the shape, size and operations at every iteration ifneeded.</p>

</div>




# Datasets & DataLoaders

Code for processing data samples can get messy and hard to maintain; we
ideally want our dataset code to be decoupled from our model training
code for better readability and modularity. PyTorch provides two data
primitives: `torch.utils.data.DataLoader` and `torch.utils.data.Dataset`
that allow you to use pre-loaded datasets as well as your own data.
`Dataset` stores the samples and their corresponding labels, and
`DataLoader` wraps an iterable around the `Dataset` to enable easy
access to the samples.

PyTorch domain libraries provide a number of pre-loaded datasets (such
as IMDb) that subclass `torch.utils.data.Dataset` and implement
functions specific to the particular data. They can be used to prototype
and benchmark your model. You can find them here: [Image
Datasets](https://pytorch.org/vision/stable/datasets.html), [Text
Datasets](https://pytorch.org/text/stable/datasets.html), and [Audio
Datasets](https://pytorch.org/audio/stable/datasets.html).

But `torchtext` deprecated since 2024 in favor of huggingface's `datasets`.

So, there are 2 options how to load a dataset: through huggingface and with custom implementation of `torch.utils.data.Dataset`.



In [5]:
%%capture
!pip install datasets

In [ ]:
import datasets

In [1]:
import os
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from glob import glob
import torch

## Loading a Dataset from HF

If the dataset has been uploaded to HF you can load it with `datasets` library and use it in training directly.

In [ ]:
dataset = datasets.load_dataset("shamotskyi/ukr_pravda_2y", split="train")
dataset = dataset.filter(lambda x: isinstance(x["ukr_text"], str))
dataset = dataset.filter(lambda x: isinstance(x["ukr_tags"], str))
dataset = dataset.map(lambda x: {"target": 1 if "війна" in x["ukr_tags"] else 0})
dataset = dataset.select_columns(["ukr_text", "target"])
dataset = dataset.class_encode_column("target")
train_test = dataset.train_test_split(test_size=0.2, seed=42, stratify_by_column="target")

ukr_pravda_2y-0.0.2.csv:  40%|####      | 168M/416M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/61629 [00:00<?, ? examples/s]

Filter:   0%|          | 0/61629 [00:00<?, ? examples/s]

Filter:   0%|          | 0/57226 [00:00<?, ? examples/s]

Map:   0%|          | 0/56131 [00:00<?, ? examples/s]

Stringifying the column:   0%|          | 0/56131 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/56131 [00:00<?, ? examples/s]

In [ ]:
train_test

DatasetDict({
    train: Dataset({
        features: ['ukr_text', 'target'],
        num_rows: 44904
    })
    test: Dataset({
        features: ['ukr_text', 'target'],
        num_rows: 11227
    })
})

## Creating a Custom Dataset for your files

Or you can implemet custom dataset class to manualy load your data from any source (e.g. from disk, stream etc).

A custom Dataset class must implement three functions:
`__init__`, `__len__`, and
`__getitem__`.

Let's first upload IMDb dataset for binary sentiment classification.


In [2]:
!wget http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz

--2025-09-21 09:23:45--  http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
Resolving ai.stanford.edu (ai.stanford.edu)... 171.64.68.10
Connecting to ai.stanford.edu (ai.stanford.edu)|171.64.68.10|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 84125825 (80M) [application/x-gzip]
Saving to: ‘aclImdb_v1.tar.gz.1’

aclImdb_v1.tar.gz.1 100%[===================>]  80.23M  55.0MB/s    in 1.5s    

2025-09-21 09:23:46 (55.0 MB/s) - ‘aclImdb_v1.tar.gz.1’ saved [84125825/84125825]



In [3]:
!tar -xzf aclImdb_v1.tar.gz

In [4]:
!ls /content/aclImdb/train

labeledBow.feat  pos	unsupBow.feat  urls_pos.txt
neg		 unsup	urls_neg.txt   urls_unsup.txt


In [5]:
!ls /content/aclImdb/test

labeledBow.feat  neg  pos  urls_neg.txt  urls_pos.txt


In [6]:
!cat /content/aclImdb/train/pos/0_9.txt

Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High's satire is much closer to reality than is "Teachers". The scramble to survive financially, the insightful students who can see right through their pathetic teachers' pomp, the pettiness of the whole situation, all remind me of the schools I knew and their students. When I saw the episode in which a student repeatedly tried to burn down the school, I immediately recalled ......... at .......... High. A classic line: INSPECTOR: I'm here to sack one of your teachers. STUDENT: Welcome to Bromwell High. I expect that many adults of my age think that Bromwell High is far fetched. What a pity that it isn't!

In [7]:
!cat /content/aclImdb/train/neg/11813_1.txt

Before Stan Laurel became the smaller half of the all-time greatest comedy team, he laboured under contract to Broncho Billy Anderson in a series of cheapies, many of which were parodies of major Hollywood features. Following a dispute with Anderson, Laurel continued the informal series of parodies at Joe Rock's smaller (and more indigent) production company.<br /><br />Most of Laurel's parody films were only mildly funny at the time, and even less funny for modern audiences who haven't seen the original movie which Laurel is parodying. 'West of Hot Dog' is a fairly generic parody of cowboy shoot-'em-ups. It's marginally a specific parody of 'West of the Pecos', an oater released two years earlier with no major actors. Since 'West of the Pecos' was never a huge success, it's difficult to see why Stan's film unit chose this particular movie as a target for their lampoonery, much less why they waited so long after its release to parody it. And where did they get that title 'West of Hot D

We can see that texts with positive label are stored in `pos` dir and with negative in `neg` one.

Let's create the custom dataset class.

In [8]:
class CustomTextDataset(Dataset):
    def __init__(self, base_dir, num_samples=None):
        if num_samples:
          pos_texts_path = glob(base_dir + "pos/*.txt")[:num_samples]
          neg_texts_path = glob(base_dir + "neg/*.txt")[:num_samples]
        else:
          pos_texts_path = glob(base_dir + "pos/*.txt")
          neg_texts_path = glob(base_dir + "neg/*.txt")
        self.texts_path = pos_texts_path + neg_texts_path

    def __len__(self):
        return len(self.texts_path)

    def __getitem__(self, idx):
        path = self.texts_path[idx]
        label = path.split('/')[-2]
        if label == "pos":
          target = 1
        elif label == "neg":
          target = 0
        else:
          raise ValueError(f"label should be 'pos' or 'neg', found {label}")
        with open(path, "r") as f:
          text = f.read()
        return text, target

In [9]:
train_dataset = CustomTextDataset("/content/aclImdb/train/")
test_dataset = CustomTextDataset("/content/aclImdb/test/")

## Tokenization

We need to map our words to ids. We will use BPE tokenizer from gpt2. You will learn more about tokenization in future lections.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def collator(batch):
  texts, labels = zip(*batch)
  return tokenizer(texts, return_tensors='pt', padding="max_length", max_length=1024, truncation=True)["input_ids"], torch.tensor(labels)

In [ ]:
tokenizer(text=tokenizer.pad_token)

{'input_ids': [50256], 'attention_mask': [1]}

In [ ]:
batch_size = 128
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collator)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collator)

In [ ]:
next(iter(train_dataloader))

(tensor([[   40,  3505,  6198,  ..., 50256, 50256, 50256],
         [   40,  1816,   284,  ..., 50256, 50256, 50256],
         [ 2061,   257, 29215,  ..., 50256, 50256, 50256],
         ...,
         [   63,   464, 24936,  ..., 50256, 50256, 50256],
         [ 4480,   845,  1310,  ..., 50256, 50256, 50256],
         [   72,   561,   588,  ..., 50256, 50256, 50256]]),
 tensor([1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1,
         1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1,
         1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0,
         0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0,
         0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0,
         1, 1, 0, 1, 1, 0, 1, 1]))

In [ ]:
next(iter(test_dataloader))[0].size()

torch.Size([128, 1024])

## Custom tokenizer using wordnet and lemming

In [10]:
import re
from collections import Counter
from typing import List, Iterable, Dict, Optional, Tuple
import torch
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet as wn

for pkg in ["averaged_perceptron_tagger", "averaged_perceptron_tagger_eng", "wordnet", "omw-1.4"]:
    try:
        nltk.data.find(f"corpora/{pkg}")
    except LookupError:
        try:
            nltk.data.find(f"taggers/{pkg}")
        except LookupError:
            nltk.download(pkg, quiet=True)

In [11]:
lemmatizer = WordNetLemmatizer()

def _to_wordnet_pos(tag: str):
    if tag.startswith("J"): return wn.ADJ
    if tag.startswith("V"): return wn.VERB
    if tag.startswith("N"): return wn.NOUN
    if tag.startswith("R"): return wn.ADV
    return wn.NOUN

_WORD_RE = re.compile(r"[A-Za-z']+")

def basic_tokenize(text: str) -> List[str]:
    return [m.group(0).lower() for m in _WORD_RE.finditer(text)]

def lemming(tokens: List[str]) -> List[str]:
    tagged = nltk.pos_tag(tokens)
    return [lemmatizer.lemmatize(tok, _to_wordnet_pos(pos)).lower() for tok, pos in tagged]

# ---------- Vocab (top-1k, remove unknowns) ----------
PAD = "<pad>"

class Vocab:
    def __init__(self, max_words: int = 1000, min_freq: int = 1):
        self.max_words = max_words
        self.min_freq = min_freq
        self.itos: List[str] = [PAD]  # PAD at 0
        self.stoi: Dict[str, int] = {PAD: 0}

    @property
    def pad_id(self) -> int:
        return 0

    @property
    def vocab_size(self) -> int:
      return len(self.itos)

    def fit(self, docs: Iterable[Tuple[str, int]]) -> None:
        cnt = Counter()
        for txt, _ in docs:
            cnt.update(lemming(basic_tokenize(txt)))
        words = [w for w, f in cnt.most_common() if f >= self.min_freq][: self.max_words]
        self.itos = [PAD] + words
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def encode_drop_oov(self, text: str) -> List[int]:
        seq = []
        for w in lemming(basic_tokenize(text)):
            idx = self.stoi.get(w)
            if idx is not None and idx != self.pad_id:  # drop OOV; never keep PAD
                seq.append(idx)
        return seq

In [12]:
# ---------- Collate: returns ONLY [B, L] LongTensor ----------
class FixedLenCollate:
    """
    Collate for raw strings:
      - encodes with vocab (dropping OOVs),
      - truncates/pads each to fixed length L,
      - returns a single LongTensor of shape [B, L].
    """
    def __init__(self, vocab: Vocab, seq_len: int):
        self.vocab = vocab
        self.seq_len = int(seq_len)

    def __call__(self, batch: List[str]) -> torch.LongTensor:
        texts, labels = zip(*batch)
        B, L = len(batch), self.seq_len
        pad_id = self.vocab.pad_id
        out = torch.full((B, L), pad_id, dtype=torch.long)

        for i, text in enumerate(texts):
            idxs = self.vocab.encode_drop_oov(text)
            if not idxs:
                continue  # stays all PADs
            # truncate then pad
            T = min(len(idxs), L)
            out[i, :T] = torch.tensor(idxs[:T], dtype=torch.long)
        return out, torch.tensor(labels)


In [13]:
vocab = Vocab(max_words=1000, min_freq=1)
vocab.fit(train_dataset)

In [14]:
collate = FixedLenCollate(vocab, seq_len=1024)
batch_size = 128
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate)

In [15]:
next(iter(train_dataloader))

(tensor([[ 13,  20, 259,  ...,   0,   0,   0],
         [ 10, 160,  36,  ...,   0,   0,   0],
         [144,  20,   3,  ...,   0,   0,   0],
         ...,
         [ 10,   2,  35,  ...,   0,   0,   0],
         [ 10, 813,   6,  ...,   0,   0,   0],
         [ 10, 225,  67,  ...,   0,   0,   0]]),
 tensor([0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
         0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0,
         1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
         1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1,
         1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0,
         1, 0, 1, 0, 0, 0, 1, 0]))

 # Build the Neural Network

Neural networks comprise of layers/modules that perform operations on
data. The [torch.nn](https://pytorch.org/docs/stable/nn.html) namespace
provides all the building blocks you need to build your own neural
network. Every module in PyTorch subclasses the
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
A neural network is a module itself that consists of other modules
(layers). This nested structure allows for building and managing complex
architectures easily.


In [16]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

## Get Device for Training

We want to be able to train our model on an
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.


In [17]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


## Define the Class

We define our neural network by subclassing `nn.Module`, and initialize
the neural network layers in `__init__`. Every `nn.Module` subclass
implements the operations on input data in the `forward` method.


In [18]:
class NeuralNetwork(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes, pad_id=0, seq_len=1024):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(seq_len * embedding_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        batch_size = x.size()[0] # x shape: (batch_size, sequence_length)
        x = self.embedding(x) # x shape: (batch_size, sequence_length, embedding_dim)
        x = x.reshape(batch_size, -1) # (batch_size, sequence_length * embedding_dim)
        logits = self.linear_relu_stack(x) # logits shape: (batch_size, num_classes)
        return logits

We create an instance of `NeuralNetwork`, and move it to the `device`,
and print its structure.


In [19]:
model = NeuralNetwork(vocab.vocab_size, 32, 2).to(device)
print(model)

NeuralNetwork(
  (embedding): Embedding(1001, 32, padding_idx=0)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=32768, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=2, bias=True)
  )
)


To use the model, we pass it the input data. This executes the model\'s
`forward`, along with some [background
operations](https://github.com/pytorch/pytorch/blob/270111b7b611d174967ed204776985cefca9c144/torch/nn/modules/module.py#L866).
Do not call `model.forward()` directly!

Calling the model on the input returns a 2-dimensional tensor with dim=0
corresponding to each output of 10 raw predicted values for each class,
and dim=1 corresponding to the individual values of each output. We get
the prediction probabilities by passing it through an instance of the
`nn.Softmax` module.


In [35]:
X = torch.randint(0, 1000, size=(1, 1024), device=device)
X

tensor([[343, 938, 591,  ..., 770, 824, 235]], device='cuda:0')

In [36]:
logits = model(X)
logits

tensor([[-0.1249, -0.1560]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [37]:
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([0], device='cuda:0')


------------------------------------------------------------------------


## Model Layers

Let\'s break down the layers in the model. To illustrate
it, we will take a sample minibatch of 3x28x28 and see
what happens to it as we pass it through the network.


In [ ]:
input_tensor = torch.randint(0, 1000, size=(1, 16))
print(input_tensor.size())

torch.Size([1, 16])


## nn.Embedding

We initialize the
[nn.Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)
layer to map each input id to a vector of fixed size. The embeddings are trainable parameters, which adjust during training.


In [ ]:
embedding = nn.Embedding(1000, 64)
emded_tensor = embedding(input_tensor)
print(emded_tensor.size())

torch.Size([1, 16, 64])


## nn.Linear

The [linear
layer](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
is a module that applies a linear transformation on the input using its
stored weights and biases.


In [ ]:
layer1 = nn.Linear(in_features=64, out_features=512)
hidden1 = layer1(emded_tensor)
print(hidden1.size())

torch.Size([1, 16, 512])


## nn.ReLU

Non-linear activations are what create the complex mappings between the
model\'s inputs and outputs. They are applied after linear
transformations to introduce *nonlinearity*, helping neural networks
learn a wide variety of phenomena.

In this model, we use
[nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
between our linear layers, but there\'s other activations to introduce
non-linearity in your model.


In [ ]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[[ 0.0788,  0.0106, -0.0360,  ..., -0.6368,  0.4684,  0.7319],
         [ 0.4176,  0.4214,  0.0066,  ...,  0.3737, -1.6466, -0.2507],
         [-0.3202, -0.4063, -1.2928,  ..., -0.4251, -0.9854,  0.6683],
         ...,
         [-0.7779, -0.6543, -0.0297,  ...,  0.3292, -0.2156, -0.0878],
         [-0.1594,  1.2548,  0.2409,  ...,  0.4036, -0.1706,  0.3089],
         [-0.1332,  0.5214, -0.9384,  ...,  0.4795, -0.2268,  0.3267]]],
       grad_fn=<ViewBackward0>)


After ReLU: tensor([[[0.0788, 0.0106, 0.0000,  ..., 0.0000, 0.4684, 0.7319],
         [0.4176, 0.4214, 0.0066,  ..., 0.3737, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.6683],
         ...,
         [0.0000, 0.0000, 0.0000,  ..., 0.3292, 0.0000, 0.0000],
         [0.0000, 1.2548, 0.2409,  ..., 0.4036, 0.0000, 0.3089],
         [0.0000, 0.5214, 0.0000,  ..., 0.4795, 0.0000, 0.3267]]],
       grad_fn=<ReluBackward0>)


## nn.Sequential

[nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html)
is an ordered container of modules. The data is passed through all the
modules in the same order as defined. You can use sequential containers
to put together a quick network like `seq_modules`.


In [ ]:
seq_modules = nn.Sequential(
    embedding,
    layer1,
    nn.ReLU(),
    nn.Linear(512, 2)
)
input_tensor = torch.randint(0, 1000, size=(1, 16))
logits = seq_modules(input_tensor)

## nn.Softmax

The last linear layer of the neural network returns [logits]{.title-ref}
- raw values in \[-infty, infty\] - which are passed to the
[nn.Softmax](https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html)
module. The logits are scaled to values \[0, 1\] representing the
model\'s predicted probabilities for each class. `dim` parameter
indicates the dimension along which the values must sum to 1.


In [ ]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)
pred_probab

tensor([[[0.0549, 0.0543],
         [0.0533, 0.0811],
         [0.0435, 0.0479],
         [0.0659, 0.0543],
         [0.0696, 0.0544],
         [0.0405, 0.0484],
         [0.0643, 0.0534],
         [0.0743, 0.0563],
         [0.0533, 0.0811],
         [0.0501, 0.0847],
         [0.0688, 0.0757],
         [0.0674, 0.0816],
         [0.0732, 0.0598],
         [0.0596, 0.0530],
         [0.0847, 0.0409],
         [0.0766, 0.0730]]], grad_fn=<SoftmaxBackward0>)

## Model Parameters

Many layers inside a neural network are *parameterized*, i.e. have
associated weights and biases that are optimized during training.
Subclassing `nn.Module` automatically tracks all fields defined inside
your model object, and makes all parameters accessible using your
model\'s `parameters()` or `named_parameters()` methods.

In this example, we iterate over each parameter, and print its size and
a preview of its values.


In [ ]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (embedding): Embedding(50257, 32, padding_idx=50256)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=32768, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=2, bias=True)
  )
)


Layer: embedding.weight | Size: torch.Size([50257, 32]) | Values : tensor([[-0.1590, -1.0427, -2.1816, -0.6034,  1.9692, -1.1177, -0.8386,  0.5583,
         -0.5914,  1.4027, -0.7724, -1.2647,  0.1198, -0.8973, -0.3004, -0.7222,
          0.4943,  1.5710, -0.1722,  0.0831,  0.9246,  1.3667, -1.7045, -1.4202,
          0.2302,  0.5153, -0.8432,  0.1983, -1.2239, -1.6851,  0.0383, -0.9540],
        [ 0.9633, -0.7406, -0.1423,  0.6034, -0.5454,  0.7473,  0.0662,  0.1175,
         -1.4253,  0.2113,  0.9296,  1.1643, -0.1586, -0.4407, -2.5192,  0.3846,
          0.6222, -0.6666,  0.8851,  0.4244,  0.3121,  0.6120,  1.4728, -1.4635,
          0.3918,

# Optimizing Model Parameters

Now that we have a model and data it\'s time to train, validate and test
our model by optimizing its parameters on our data. Training a model is
an iterative process; in each iteration the model makes a guess about
the output, calculates the error in its guess (*loss*), collects the
derivatives of the error with respect to its parameters, and **optimizes** these
parameters using gradient descent. For a more detailed walkthrough of
this process, check out this video on [backpropagation from
3Blue1Brown](https://www.youtube.com/watch?v=tIeHLnjs5U8).


## Hyperparameters

Hyperparameters are adjustable parameters that let you control the model
optimization process. Different hyperparameter values can impact model
training and convergence rates ([read
more](https://pytorch.org/tutorials/beginner/hyperparameter_tuning_tutorial.html)
about hyperparameter tuning)

We define the following hyperparameters for training:

:   -   **Number of Epochs** - the number of times to iterate over the
        dataset
    -   **Batch Size** - the number of data samples propagated through
        the network before the parameters are updated
    -   **Learning Rate** - how much to update models parameters at each
        batch/epoch. Smaller values yield slow learning speed, while
        large values may result in unpredictable behavior during
        training.


In [20]:
learning_rate = 5e-2
epochs = 10

## Optimization Loop

Once we set our hyperparameters, we can then train and optimize our
model with an optimization loop. Each iteration of the optimization loop
is called an **epoch**.

Each epoch consists of two main parts:

:   -   **The Train Loop** - iterate over the training dataset and try
        to converge to optimal parameters.
    -   **The Validation/Test Loop** - iterate over the test dataset to
        check if model performance is improving.

Let\'s briefly familiarize ourselves with some of the concepts used in
the training loop. Jump ahead to see the
`full-impl-label`{.interpreted-text role="ref"} of the optimization
loop.

## Loss Function

When presented with some training data, our untrained network is likely
not to give the correct answer. **Loss function** measures the degree of
dissimilarity of obtained result to the target value, and it is the loss
function that we want to minimize during training. To calculate the loss
we make a prediction using the inputs of our given data sample and
compare it against the true data label value.

Common loss functions include
[nn.MSELoss](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html#torch.nn.MSELoss)
(Mean Square Error) for regression tasks, and
[nn.NLLLoss](https://pytorch.org/docs/stable/generated/torch.nn.NLLLoss.html#torch.nn.NLLLoss)
(Negative Log Likelihood) for classification.
[nn.CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html#torch.nn.CrossEntropyLoss)
combines `nn.LogSoftmax` and `nn.NLLLoss`.

We pass our model\'s output logits to `nn.CrossEntropyLoss`, which will
normalize the logits and compute the prediction error.


In [21]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

## Optimizer

Optimization is the process of adjusting model parameters to reduce
model error in each training step. **Optimization algorithms** define
how this process is performed (in this example we use Stochastic
Gradient Descent). All optimization logic is encapsulated in the
`optimizer` object. Here, we use the SGD optimizer; additionally, there
are many [different
optimizers](https://pytorch.org/docs/stable/optim.html) available in
PyTorch such as ADAM and RMSProp, that work better for different kinds
of models and data.

We initialize the optimizer by registering the model\'s parameters that
need to be trained, and passing in the learning rate hyperparameter.


In [22]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

Inside the training loop, optimization happens in three steps:

:   -   Call `optimizer.zero_grad()` to reset the gradients of model
        parameters. Gradients by default add up; to prevent
        double-counting, we explicitly zero them at each iteration.
    -   Backpropagate the prediction loss with a call to
        `loss.backward()`. PyTorch deposits the gradients of the loss
        w.r.t. each parameter.
    -   Once we have our gradients, we call `optimizer.step()` to adjust
        the parameters by the gradients collected in the backward pass.


## Full Implementation

We define `train_loop` that loops over our optimization code, and
`test_loop` that evaluates the model\'s performance against our test
data.


In [23]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    train_loss, correct = 0, 0
    num_batches = len(dataloader)
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        X = X.to(device)
        y = y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        train_loss += loss.item()
        correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    train_loss /= num_batches
    correct /= size
    print(f"Train Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {train_loss:>8f} \n")


def validation_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    validation_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)
            validation_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    validation_loss /= num_batches
    correct /= size
    print(f"Validation Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {validation_loss:>8f} \n")
    return validation_loss

We initialize the loss function and optimizer, and pass it to
`train_loop` and `test_loop`. Feel free to increase the number of epochs
to track the model\'s improving performance.


In [24]:
import numpy as np

device="cuda"

model = NeuralNetwork(vocab.vocab_size, 32, 2).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

min_val_loss = np.inf

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    print("Validation:")
    val_loss = validation_loop(test_dataloader, model, loss_fn)
    if val_loss > min_val_loss:
      break
    min_val_loss = min(val_loss, min_val_loss)

print("Done!")

Epoch 1
-------------------------------
Train Error: 
 Accuracy: 54.1%, Avg loss: 0.688704 

Validation:
Validation Error: 
 Accuracy: 57.8%, Avg loss: 0.680764 

Epoch 2
-------------------------------
Train Error: 
 Accuracy: 68.2%, Avg loss: 0.613122 

Validation:
Validation Error: 
 Accuracy: 58.7%, Avg loss: 0.678165 

Epoch 3
-------------------------------
Train Error: 
 Accuracy: 78.7%, Avg loss: 0.456327 

Validation:
Validation Error: 
 Accuracy: 58.4%, Avg loss: 0.758671 

Done!


# Save and Load the Model

In this section we will look at how to persist model state with saving,
loading and running model predictions.


Saving and Loading Model Weights
================================

PyTorch models store the learned parameters in an internal state
dictionary, called `state_dict`. These can be persisted via the
`torch.save` method:


In [ ]:
torch.save(model.state_dict(), 'model_weights.pth')

To load model weights, you need to create an instance of the same model
first, and then load the parameters using `load_state_dict()` method.

In the code below, we set `weights_only=True` to limit the functions
executed during unpickling to only those necessary for loading weights.
Using `weights_only=True` is considered a best practice when loading
weights.


In [ ]:
model = NeuralNetwork(tokenizer.vocab_size, 64, 2) # we do not specify ``weights``, i.e. create untrained model
model.load_state_dict(torch.load('model_weights.pth', weights_only=True))
model.eval()

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>be sure to call <code>model.eval()</code> method before inferencing to set the dropout and batch normalization layers to evaluation mode. Failing to do this will yield inconsistent inference results.</p>

</div>



Saving and Loading Models with Shapes
=====================================

When loading model weights, we needed to instantiate the model class
first, because the class defines the structure of a network. We might
want to save the structure of this class together with the model, in
which case we can pass `model` (and not `model.state_dict()`) to the
saving function:


In [ ]:
torch.save(model, 'model.pth')

We can then load the model as demonstrated below.

As described in [Saving and loading
torch.nn.Modules](https://pytorch.org/docs/main/notes/serialization.html#saving-and-loading-torch-nn-modules),
saving `state_dict` is considered the best practice. However, below we
use `weights_only=False` because this involves loading the model, which
is a legacy use case for `torch.save`.


In [ ]:
model = torch.load('model.pth', weights_only=False),

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>This approach uses Python <a href="https://docs.python.org/3/library/pickle.html">pickle</a> module when serializing the model, thus it relies on the actual class definition to be available when loading the model.</p>

</div>

